# Ingest Human Frame Annotations

This notebook reads completed pilot and validation CSVs, checks hierarchical label validity, derives frame labels, and writes clean human annotation tables for the classifier stage.


In [ ]:
from __future__ import annotations

from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "01_classification":
    PROJECT_ROOT = PROJECT_ROOT.parents[1]

CLASSIFICATION_DIR = PROJECT_ROOT / "data/interim/lsc/classification"
HUMAN_DIR = CLASSIFICATION_DIR / "human_annotation"
OUTPUT_DIR = CLASSIFICATION_DIR / "human_labels"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PILOT_COMPLETED = HUMAN_DIR / "frame_pilot_annotation_completed.csv"
VALIDATION_COMPLETED = HUMAN_DIR / "frame_validation_annotation_completed.csv"

EXPECTED_COLUMNS = [
    "annotation_id",
    "context_id",
    "analysis_unit",
    "lsc_year",
    "raw_form",
    "target_sentence_plus_adjacent",
    "substantive_target_discourse",
    "clinical_frame_present",
    "lived_experience_frame_present",
    "confidence",
    "annotation_round",
    "codebook_version",
]
CONFIDENCE_VALUES = {"high", "medium", "low"}


## Load Completed Sheets

Save your annotated CSVs with `_completed.csv` filenames before running this notebook.

In [ ]:
missing = [path for path in [PILOT_COMPLETED, VALIDATION_COMPLETED] if not path.exists()]
if missing:
    print("Completed annotation files not found yet:")
    for path in missing:
        print(f"- {path.relative_to(PROJECT_ROOT)}")
    raise SystemExit("Add completed CSVs, then rerun.")

try:
    pilot = pd.read_csv(PILOT_COMPLETED, encoding="utf-8")
    validation = pd.read_csv(VALIDATION_COMPLETED, encoding="utf-8")
except UnicodeDecodeError as error:
    raise UnicodeDecodeError(error.encoding, error.object, error.start, error.end, "Completed annotation CSVs must be saved as UTF-8.") from error

human = pd.concat([pilot, validation], ignore_index=True)

missing_columns = sorted(set(EXPECTED_COLUMNS) - set(human.columns))
if missing_columns:
    raise ValueError(f"Missing expected columns: {missing_columns}")

print(f"Loaded human annotations: {len(human):,}")


## Validate and Derive Frames

The Stage-0 sufficiency gate is labelled for every row. Clinical and lived-experience labels are required only for substantive rows and must be `NA` for non-substantive rows.


In [ ]:
def parse_bool_or_na(value: object) -> bool | pd.NA:
    if pd.isna(value):
        return pd.NA
    text = str(value).strip().lower()
    if text in {"", "na", "n/a", "none", "nan"}:
        return pd.NA
    if text in {"true", "t", "yes", "y", "1"}:
        return True
    if text in {"false", "f", "no", "n", "0"}:
        return False
    return pd.NA

for column in ["substantive_target_discourse", "clinical_frame_present", "lived_experience_frame_present"]:
    human[column] = human[column].map(parse_bool_or_na).astype("boolean")

invalid_substantive = human.loc[human["substantive_target_discourse"].isna(), ["annotation_id", "substantive_target_discourse"]]
if not invalid_substantive.empty:
    raise ValueError(f"Invalid or missing Stage-0 labels: {invalid_substantive.head(20).to_dict('records')}")

substantive = human["substantive_target_discourse"].eq(True)
non_substantive = human["substantive_target_discourse"].eq(False)
invalid_stage1_substantive = human.loc[
    substantive & (human["clinical_frame_present"].isna() | human["lived_experience_frame_present"].isna()),
    ["annotation_id", "clinical_frame_present", "lived_experience_frame_present"],
]
if not invalid_stage1_substantive.empty:
    raise ValueError(f"Missing Stage-1 labels for substantive rows: {invalid_stage1_substantive.head(20).to_dict('records')}")

invalid_stage1_non_substantive = human.loc[
    non_substantive & (human["clinical_frame_present"].notna() | human["lived_experience_frame_present"].notna()),
    ["annotation_id", "clinical_frame_present", "lived_experience_frame_present"],
]
if not invalid_stage1_non_substantive.empty:
    raise ValueError(f"Clinical/lived labels must be NA for non-substantive rows: {invalid_stage1_non_substantive.head(20).to_dict('records')}")

human["confidence"] = human["confidence"].astype(str).str.strip().str.lower()
invalid_confidence = human.loc[~human["confidence"].isin(CONFIDENCE_VALUES), ["annotation_id", "confidence"]]
if not invalid_confidence.empty:
    raise ValueError(f"Invalid confidence values: {invalid_confidence.head(20).to_dict('records')}")


def derive_frame(row: pd.Series) -> str:
    if not bool(row["substantive_target_discourse"]):
        return "non_substantive_or_insufficient"
    clinical = bool(row["clinical_frame_present"])
    lived = bool(row["lived_experience_frame_present"])
    if clinical and lived:
        return "mixed"
    if clinical:
        return "clinical_only"
    if lived:
        return "lived_only"
    return "substantive_other"

human["derived_frame"] = human.apply(derive_frame, axis=1)

duplicate_ids = human["annotation_id"].duplicated().sum()
duplicate_contexts = human["context_id"].duplicated().sum()
if duplicate_ids or duplicate_contexts:
    raise ValueError(f"Duplicate IDs found: annotation_id={duplicate_ids}, context_id={duplicate_contexts}")

human.groupby(["annotation_round", "analysis_unit", "derived_frame"]).size()


## Save Clean Human Labels

In [ ]:
human_path = OUTPUT_DIR / "frame_human_labels.csv"
pilot_path = OUTPUT_DIR / "frame_human_pilot_labels.csv"
validation_path = OUTPUT_DIR / "frame_human_validation_labels.csv"

human.to_csv(human_path, index=False)
human.loc[human["annotation_round"].eq("pilot")].to_csv(pilot_path, index=False)
human.loc[human["annotation_round"].eq("validation")].to_csv(validation_path, index=False)

print("Wrote clean human labels:")
for path in [human_path, pilot_path, validation_path]:
    print(f"- {path.relative_to(PROJECT_ROOT)}")